# NN Training to Substitute LUT

## Setup
Set-up Keras (in order to use nnodely) in configuration "torch" to use PyTorch.
Import libraries to navigate repository, generate figures and open `.np` files (LUT table).

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

%load_ext autoreload
%autoreload 2

## Convert `nc` to `csv` & Rename Variable
To better visualize the LUT, convert the `.nc` file into a `.csv` (step not strictly necessary as we will use `nnodely` directly with the `.nc` file).

| Original Name | New Name |
| :--- | :--- |
| `__xarray_dataarray_variable__` | `activation_value` |
| `min_torque` | `torque_min` |
| `max_torque` | `torque_max` |

In [ ]:
from src.lut import LUT

PATH_FILE_NC = "models/W_all/lut_W_all_2.nc"
PATH_FILE_CSV = "models/W_all/lut_W_all_2.csv"

OLD_VAR_NAME = "max_torque"
NEW_VAR_NAME = "torque_max"

my_lut = LUT(PATH_FILE_NC)

my_lut.rename_variable(OLD_VAR_NAME, NEW_VAR_NAME)
my_lut.to_csv(PATH_FILE_CSV)

## LUT Upload
Open the `.nc` file and remove the NaN.
Build the LUT Matrix starting from the `.nc` file.
The numer of rows of the LUT matrix should be the valid rows of CSV divided by 6.
In the `.nc` file, values are stored as `float64`, but PyTorch will convert it into `float32`.

In [ ]:
from src.lut import LUT

# modify this if you want to upload another np
#LUT_PATH = PATH_FILE_NC
LUT_PATH = "models/W_all/lut_W_all_2.nc"

my_lut = LUT(LUT_PATH)
my_lut.load_data()

## LUT Visualization
Heat-map to visualize the intensity of the muscle.
Insert negative values for `ANGLE_REQ` or `TAU_REQ` if you wanto to analize the general case.

In [ ]:
ANGLE_REQ = 45.0
TAU_REQ = 0.8

#my_lut.plot_heatmaps()
my_lut.plot_heatmaps(considered_W=0.3, angle_req=ANGLE_REQ, tau_req=TAU_REQ)

## NN Setup
We need to define the number of layers and of neurons of the Neural Network.
The input is $x_{in}=[\theta, \tau_{norm}]^T$ and output is $u_{out}=[u(0),\dots, u(5)]^T$.

Note that it is necessary to pre-process the data. Due to the fact that $\tau_{norm} \in [0,1]$ and $\theta \in [0°,130°]$, variation of $\tau_{norm}$ will be ininfluent to the NN weight variation. Therefore, we neet to normalize the inputs of the dataset.

In [ ]:
from src.nn_model_from_lut import NN_model_from_LUT

NN_NAME = "comb_lut_W_try"
NUM_HIDDEN_LAYERS = 2
HIDDEN_UNITS = 8
ACTIVATION_MODE = "relu"

HOME_DIR = "models/W_try"
SAVE_PATH_PLOTS = f"{HOME_DIR}/{NN_NAME}/plots"

my_nn_model = NN_model_from_LUT(NN_NAME, my_lut)

my_nn_model.define_structure(num_hidden_layers = NUM_HIDDEN_LAYERS,
                             hidden_units = HIDDEN_UNITS,
                             activation = ACTIVATION_MODE,
                             show_table = True, 
                             save_path_plots = SAVE_PATH_PLOTS)

## NN Training

shuffle=True,       # Rimescola i batch a ogni epoca per evitare bias di ordinamento

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

my_nn_model.train(epochs = EPOCHS,
                  batch_size = BATCH_SIZE,
                  lr = LEARNING_RATE,
                  show_plots = False,
                  save_path_plots = SAVE_PATH_PLOTS)

## NN Export Model

In [ ]:
SAVE_PATH_EXPORT_MODEL = f"{HOME_DIR}/{NN_NAME}"

my_nn_model.export(SAVE_PATH_EXPORT_MODEL)

## Error Heatmap
$\Delta u = u_{LUT}- u_{NN}$

In [ ]:

#my_nn_model.plot_heatmaps()
#my_nn_model.plot_error_heatmaps()
my_nn_model.plot_error_heatmaps(considered_W = 0.2,
                                save_path=SAVE_PATH_PLOTS)